# 解答例（教員用）　第11回
***
> このノートブックは **模範解答の一例**です．コードの空欄（`___`）を埋め，探索は複数ラウンドに分けて実行し，解答欄には実際に得られた出力の数値を根拠に書いています．
> 乱数を使う箇所は固定していますが，ライブラリのバージョンや実行環境によって数値が **小数第2〜3位ほどずれる**ことがあります．探索型の問題なので，**別の探索経路・別の最良設定でも，根拠が筋の通っていれば正解**です．

# 第11回　教師なし学習
***
> **前提**: 第6回までの教師あり学習に続き，ラベルなしデータの分析を学びます。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（機械学習）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
> | **【探索】** | 最適設定を自分で探す | **仮説 → 試行 → 記録 → 次の仮説** を何度も繰り返す。全試行が自動で履歴に残る |
> | **【指示書】** | 知識を LLM に渡す | 実験で得た知識を、LLM が使える「指示書」に書き起こし、LLM の出力を自分のログで採点する |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

---

## 🧪 この回の進め方：最適な設定は「自分で」探す

この回では **「正解のパラメータ」は教えません**。答えは、あなたが何度も試して見つけます。

```
  ①仮説を立てる → ②試す（1回1行の記録が自動で残る） → ③結果を見て仮説を直す → ①へ戻る
```

### 試行錯誤の記録帳 `Lab`（最初のコードセルで用意済み。中身を読んで使い方を理解すること）

| 使い方 | 何をするか |
|---|---|
| `lab.df()` | これまでの全試行を表で見る（試行番号・設定・スコア・仮説） |
| `lab.plot(x="k", group=...)` | 設定とスコアの関係、ベスト更新の推移をグラフにする |
| `lab.report()` | 探索の自己診断（回数・範囲・端に張り付いていないか・仮説を書いたか） |
| `lab.final_test(...)` | **テストデータでの最終評価。1回しか実行できない** |
| `lab.save()` | 全履歴を CSV に保存する |

### ルール（ここが「AIの理解」の中心）

1. **探索に使ってよいのはラベル無しで測れる指標（シルエットスコアなど）だけ**。テストデータは最後に **1回だけ**（見てから設定を変えたらデータリーク）。
2. **試行のたびに「仮説」を書く**（何を確かめたい試行か）。やみくもな総当たりは評価されません。
3. **粗く → 細かく**：まず桁・範囲を広く振って良さそうな所を掴み、次にその周辺を細かく調べる。
4. **ベストが探索範囲の端に来たら、範囲を広げて確かめる**。
5. 上位の設定同士の差が、ばらつきより小さいなら「差がある」とは言えない。
   ※教師なし学習の「テスト」に当たるのは、**正解ラベルとの一致度（ARI）**です。ラベルは最後に **1回だけ** 見ます（ラベルを見て手法を選ぶと、ラベルなしで使えない手法選びになる）。
6. **実験の前に予測を書く**。予測と結果のズレが、あなたが得た「知識」の証拠になる。
7. 最後に、得た知識を **LLM への指示書**にまとめる（最終問題）。

### 評価の観点

| 観点 | 見られること |
|---|---|
| 試行の量と質 | 目安回数以上・仮説が毎回ある・粗→細の流れがある |
| 結果の読み取り | 外れた仮説から何を学んだか（試行番号 `#n` を引用して書く） |
| 検証の作法 | テストは1回だけ／検証スコアとテストの差を正しく解釈している |
| 知識の言語化 | 自分の実験結果を、LLM が実行できる指示書に落とせている |

## 目次
1. k-means クラスタリング
2. クラスタ数の評価
3. PCA による次元削減
4. 2次元可視化
5. 実験で得た知識を LLM への指示書にする【指示書】

---

## この回で学ぶこと

### 教師あり学習 vs 教師なし学習

これまでの学習（第1〜10回）は，すべて「正解ラベル」がある**教師あり学習**だった。

```
教師あり学習: データ + 正解ラベル → モデルが「入力→出力」の関係を学習
教師なし学習: データのみ           → データ内の隠れた構造・パターンを発見
```

現実には「正解ラベルがない」データの方が圧倒的に多い。センサーデータ，SNS投稿，画像，遺伝子発現データなど，ラベル付けコストが高いデータには教師なし学習が不可欠だ。

### k-means クラスタリング

k-means の仕組みを直感的に理解しよう：

1. ランダムにk個の「重心（centroid）」を配置
2. 各データ点を最も近い重心に割り当て（クラスタを形成）
3. 各クラスタの重心を再計算
4. 変化がなくなるまで2〜3を繰り返す

**重要な特性**：
- 初期値依存性：`random_state` を固定しないと毎回結果が異なる
- k は事前に決める必要がある（最大の欠点）
- 球形クラスタ向け：細長い，三日月形などのクラスタは苦手
- スケールに敏感：`StandardScaler` で正規化が必須

### シルエットスコアとは

シルエットスコアは「各データ点が自分のクラスタに適切に属しているか」を -1〜1 の値で評価する指標だ：
- **値が1に近い**: そのデータ点は自分のクラスタの中心に近く，他のクラスタから遠い → 良い分類
- **値が0に近い**: クラスタの境界付近にある
- **値が負**: 本来は別のクラスタに属すべきかもしれない

### 主成分分析（PCA）とは

高次元データ（例：30変数）を低次元（2次元）に圧縮する手法だ。単純に変数を削除するのではなく，**データの分散が最大になる方向（主成分）を見つけ**，その方向に投影する。

活用場面：
- **可視化**: 30次元のデータを2次元にして散布図で確認
- **前処理**: 次元を減らして計算コストを削減，ノイズ除去
- **多重共線性の解決**: 相関する変数をまとめる

> **卒業研究での活用例**: テキストデータの単語ベクトル（数千次元）を PCA で2〜3次元に圧縮し，似た意味の単語をクラスタリングする研究はよく見られる。

---

## 📚 将来の自分のためのメモ — 大学研究での応用


### どんな研究で使われるか

| 分野 | 具体例 |
|---|---|
| マーケティング・経営 | 顧客の購買行動データから**セグメント**（k-means）を発見 |
| 生物学 | 遺伝子発現・タンパク質プロファイルの**サブタイプ分類** |
| 社会学・政治学 | アンケート回答者のタイプ分け，世論データのパターン探索 |
| NLP・言語学 | 単語ベクトルを PCA で可視化し，意味空間の構造を分析 |
| 画像・信号処理 | 高次元特徴を PCA で圧縮してから分類（次元の呪いの緩和） |

### 応用できる場面

- **正解ラベルがない**（またはラベル付けコストが高い）探索的分析
- データの**全体構造を把握**したい（可視化，異常検知の前段階）
- 変数が多すぎて可視化・計算が困難 → **PCA で次元削減**
- 仮説生成：「クラスタ A と B で何が違うか」を後から調べる

### 応用しにくい・向かない場面

- **正解ラベルがある予測問題**（教師あり学習の方が直接的に精度を出せる）
- クラスタが**細長い・三日月形**など非球形（k-means の前提を満たさない）→ DBSCAN，GMM など
- **クラスタ数 k が理論的に決まっている**場合でも，k-means は初期値依存で結果がぶれることがある
- PCA は**線形な圧縮**なので，強い非線形構造は取りこぼす（t-SNE，UMAP は可視化向きだが距離の保存は別問題）
- 「クラスタ = 意味のあるグループ」と**自動的には言えない**（解釈は研究者の追加分析が必要）

### 研究で報告するときのポイント

1. **k の選定根拠**（エルボー法，シルエットスコア）を図付きで示す
2. PCA では**累積寄与率**（何 % の分散を何次元で説明したか）を報告
3. クラスタごとの特徴（平均値，代表サンプル）を示し，**解釈可能性**を議論する
4. `StandardScaler` を使ったことを明記（スケール無視は k-means で致命的）

### 関連する発展トピック（調べてみると良い）

- 階層的クラスタリング，DBSCAN，Gaussian Mixture Model
- t-SNE / UMAP（非線形可視化）
- オートエンコーダ（深層学習による非線形次元削減）

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.datasets import load_iris, load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler


# ==== 試行錯誤の記録帳（この回の全実験がここに溜まる。中身を読んで使い方を理解しておくこと）====
class Lab:
    """試行錯誤の記録帳。何回でも log() でき、表・グラフ・診断レポートにできる。
    ・スコア（score）は「検証用スコア（CV）」だけを記録する。テストデータは final_test() で1回だけ。"""

    def __init__(self, name, score_name="cv_score", higher_is_better=True):
        self.name, self.score_name, self.hib = name, score_name, higher_is_better
        self.rows, self.test_result = [], None

    def seen(self, params):
        return any(r["params"] == params for r in self.rows)

    def log(self, hypothesis, params, score, **extra):
        """1試行を記録する。同じ params を再度記録しようとすると無視する（重複の水増し防止）。"""
        if self.seen(params):
            print(f"  (skip) 既に試した設定です: {params}")
            return
        self.rows.append(dict(trial=len(self.rows) + 1, hypothesis=hypothesis.strip(),
                              params=dict(params), score=float(score), extra=extra))
        best = self.best()["trial"] == len(self.rows)
        print(f"  #{len(self.rows):02d} {params}  {self.score_name}={score:.4f}"
              + "".join(f"  {k}={v}" for k, v in extra.items()) + ("  ★ベスト更新" if best else ""))

    def df(self):
        import pandas as pd
        return pd.DataFrame([{"trial": r["trial"], **r["params"], self.score_name: round(r["score"], 4),
                              **r["extra"], "hypothesis": r["hypothesis"]} for r in self.rows])

    def best(self):
        return (max if self.hib else min)(self.rows, key=lambda r: r["score"])

    def plot(self, x, group=None, logx=True, query=None):
        """パラメータ x とスコアの関係（group ごとに線を分ける）と、試行回数ごとのベスト更新の推移。"""
        import matplotlib.pyplot as plt
        d = self.df()
        dd = d.query(query) if query else d
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        for g, sub in (dd.groupby(group) if group else [("all", dd)]):
            sub = sub.sort_values(x)
            ax[0].plot(sub[x], sub[self.score_name], "o-", label=str(g))
        if logx and dd[x].dtype.kind in "if" and (dd[x] > 0).all(): ax[0].set_xscale("log")
        ax[0].set_xlabel(x); ax[0].set_ylabel(self.score_name); ax[0].legend(); ax[0].set_title("Parameter vs. validation score")
        cur = d[self.score_name].cummax() if self.hib else d[self.score_name].cummin()
        ax[1].step(d["trial"], cur, where="post"); ax[1].plot(d["trial"], d[self.score_name], "x", alpha=.5)
        ax[1].set_xlabel("Trial number"); ax[1].set_title("Best score so far (flat = plateau)")
        plt.tight_layout(); plt.show()

    def report(self, min_trials=15):
        """探索の質の自己診断：回数・カバー範囲・端に張り付いていないか・仮説を書いたか。"""
        import numpy as np
        d, n = self.df(), len(self.rows)
        print(f"■ 試行回数: {n}（目安 {min_trials} 回以上） → {'OK' if n >= min_trials else 'まだ足りない'}")
        keys = list(self.rows[0]["params"]) if n else []
        b = self.best()
        print(f"■ ベスト: trial#{b['trial']} {b['params']}  {self.score_name}={b['score']:.4f}")
        for k in keys:
            vals = [r["params"][k] for r in self.rows]
            uniq = sorted(set(map(str, vals)))
            line = f"■ {k}: {len(uniq)} 種類を試した"
            nums = [v for v in vals if isinstance(v, (int, float)) and not isinstance(v, bool)]
            if len(nums) == len(vals) and len(set(nums)) > 2:
                edge = k != "seed" and b["params"][k] in (min(nums), max(nums))
                line += f"（範囲 {min(nums):g}〜{max(nums):g}）" + (" ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう" if edge else "")
            print(line)
        w = sum(1 for r in self.rows if len(r["hypothesis"]) >= 8)
        print(f"■ 仮説を書いた試行: {w}/{n}" + ("" if w == n else " ⚠ 何を確かめる試行か毎回書こう"))
        if n >= 6:
            third = n // 3
            print(f"■ 前半の最高 {d[self.score_name][:third].max():.4f} → 後半の最高 {d[self.score_name][-third:].max():.4f}")

    def final_test(self, score_fn):
        """テストデータでの最終評価。1回しか実行できない（2回目以降はエラー）。"""
        if self.test_result is not None:
            raise RuntimeError(f"テストは既に使用済みです（結果 {self.test_result:.4f}）。テストを見て設定を変えるのはデータリークです。")
        self.test_result = float(score_fn())
        print(f"最終テストスコア = {self.test_result:.4f}   （検証ベスト = {self.best()['score']:.4f}）")
        return self.test_result

    def save(self, path=None):
        path = path or f"{self.name}_log.csv"
        self.df().to_csv(path, index=False, encoding="utf-8-sig"); print("保存:", path)


## 問題1　k-means クラスタリング　【説明】
***

### Iris データセットについて

Iris データセットは機械学習の入門で最もよく使われるデータセットだ：
- 150件のアヤメの花のデータ
- 4特徴量：がく片の長さ/幅，花びらの長さ/幅（単位：cm）
- 3クラス：Setosa / Versicolor / Virginica

今回は「正解ラベル（品種）を使わずに」，特徴量だけから3つのグループを自動発見できるかを試す。発見したクラスタが実際の品種と一致すれば，この手法の有効性が確認できる。

### なぜ標準化が必要か

Iris の4特徴量はどれもcm単位で似たスケールだが，k-means はユークリッド距離を使うため，スケールが大きい変数の影響が過大になる。**標準化は k-means の前処理として必須**だと覚えておこう。

### クラスタラベルの注意点

k-means のラベルは 0, 1, 2 が割り当てられるが，この番号と品種の名前は対応していない。例えば「クラスタ0 = Setosa」とは限らず，実行するたびに対応が変わりうる（番号に意味はない）。

### 課題

下のコードセルは、Iris を標準化して `KMeans` でクラスタリングし、シルエットスコアを出すところまで **完成済み**です。今回はコードを書くのではなく、**1行ずつ「何をしているか」を読み解く**のが目的です。

各行の `# 説明:` の右に、その行が何をしているかを**自分の言葉で**書いてください（コードは変更しないこと）。書き終えたらセルを実行し、エラーなくシルエットスコアが出ることを確認してください。

説明を書くときは、次の問いを意識してください：

- なぜ k-means の前に `StandardScaler` で標準化するのか？（k-means が使う「距離」と関係づけて）
- `fit_predict` は「学習」と「割り当て」のどちらも行うが、具体的に何を返しているのか？
- `silhouette_score` の値が高い／低いと、クラスタリングについて何が言えるのか？

> **設計判断（一言で）**: `KMeans(n_clusters=3, ...)` の `3` はどこから来た数字でしょうか。Iris の正解ラベルを「使わずに」この `3` を決めるにはどうすればよいか、解答用コードセルに1〜2文で書いてください（次の問題2の伏線です）。

### まずデータの中身を見てみよう（実行するだけ）

Iris（アヤメ）データです．コードを変更せずに実行して，**何が入っているか**を確認してから，次のセルの説明を書いてください．

In [2]:
# === データの中身を確認する（実行するだけ。コードは変更しない）===
_iris = load_iris()
print("特徴量の名前:", _iris.feature_names)
print("品種の名前   :", list(_iris.target_names))
print("データの形   :", _iris.data.shape, "（件数, 特徴量数）")
print("\n先頭 5 件の特徴量（cm）:")
print(_iris.data[:5])
print("\n先頭 5 件の品種ラベル（0/1/2）:", _iris.target[:5], "  ← クラスタリングでは使わない")
print("\n各特徴量の標準偏差:", _iris.data.std(axis=0).round(2))

特徴量の名前: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
品種の名前   : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
データの形   : (150, 4) （件数, 特徴量数）

先頭 5 件の特徴量（cm）:
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]

先頭 5 件の品種ラベル（0/1/2）: [0 0 0 0 0]   ← クラスタリングでは使わない

各特徴量の標準偏差: [0.83 0.43 1.76 0.76]


In [3]:
# 各行の # 説明: に自分の言葉で意味を書いてください（コードは変更しない）
# 説明は AI に書かせず、自分で書くこと

iris = load_iris()                                        # 説明: scikit-learn 付属の Iris（アヤメ）データセットを読み込む．特徴量 4 つと品種ラベルが入った辞書風のオブジェクト．
X_iris = iris.data                                        # 説明: 特徴量（がく片・花びらの長さと幅）だけを取り出す．形状は (150, 4)．品種ラベルは使わない．

scaler = StandardScaler()                                 # 説明: 各特徴量を平均 0・標準偏差 1 にそろえる道具を作る．k-means はユークリッド距離を使うので，スケールの大きい特徴量に距離が引きずられないよう標準化が必要．
X_scaled = scaler.fit_transform(X_iris)                   # 説明: 各列の平均・標準偏差を計算（fit）し，その値で全データを標準化（transform）する．

kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)  # 説明: クラスタ数（分けるグループ数）を 3 にする．random_state=0 は初期値の乱数を固定して結果を再現可能にし，n_init=10 は初期値を変えて10回試して最良を採る．
labels = kmeans.fit_predict(X_scaled)                     # 説明: クラスタリングを学習（fit）し，各データがどのクラスタに属するかのラベル（0,1,2 の配列，長さ 150）を返す．

print("先頭10件のクラスタラベル:", labels[:10])           # 説明: 最初の 10 件のクラスタ番号を表示する．番号は品種名と対応しない（番号自体に意味は無い）．
score = silhouette_score(X_scaled, labels)                # 説明: 第1引数はデータ，第2引数はクラスタラベル．各点について「同じクラスタ内の近さ」と「他クラスタの遠さ」から -1〜1 のスコアを出し，その平均を返す．高いほどクラスタがよく分離している．
print(f"シルエットスコア: {score:.4f}")


先頭10件のクラスタラベル: [1 1 1 1 1 1 1 1 1 1]
シルエットスコア: 0.4599


In [4]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) なぜ k-means の前に標準化するのか（k-means が使う「距離」と関連づけて）
answer_1_a = """
k-means は点とクラスタ中心のユークリッド距離で割り当てるので、値の範囲が大きい特徴量ほど距離を支配する。標準化で各特徴量を平均 0・分散 1 にそろえ、どの特徴量も公平に距離へ寄与させるため。（Iris は全部 cm 単位で似たスケールだが、一般には必須の習慣。）
"""

# (1-b) `fit_predict` は具体的に何を返しているか
answer_1_b = """
学習（fit）と割り当て（predict）を続けて行い、各データ（150 件）が属するクラスタ番号（0,1,2 の整数配列、長さ 150）を返す。
"""

# (1-c) `silhouette_score` が高い／低いとクラスタリングについて何が言えるか
answer_1_c = """
シルエットスコア（-1〜1）が高いほど、各点が自分のクラスタに近く、他のクラスタから遠い＝クラスタがよく分離している。1 に近いほど良く、0 付近はクラスタの境界上に点がいて重なっている、負はクラスタが誤って割り当てられている可能性がある。
"""

# (1-d) 設計判断：`n_clusters=3` の `3` はどこから来たか／正解ラベルを使わずに決めるには（1〜2文）
answer_1_d = """
3 は Iris が 3 品種だと知っているから（正解ラベルから）。ラベルを使わずに決めるには、k を変えながらエルボー法（inertia）やシルエットスコアなどのラベル不要の指標を比べて k を選べばよい。
"""



## 問題2　最適なクラスタリング設定を自分で探す　【探索】
***

### 「正解ラベルが無い」ときの探索

教師なし学習には accuracy のような「正解との一致率」がありません。そこで **ラベル無しで測れる指標** を物差しにして、設定を探します。

| 指標 | 意味 | 注意 |
|---|---|---|
| `silhouette`（**探索の物差し**） | クラスタ内が近く、他クラスタと遠いほど高い（-1〜1） | 標準化した空間で統一して測る（設定間で公平に比べるため） |
| `inertia`（kmeans のみ） | クラスタ内誤差の合計 | k を増やせば**必ず**下がる。単独では選べない（エルボー法で使う） |
| `ARI`（**最後に1回だけ**） | 正解の品種との一致度（1 で完全一致） | 現実のデータには無い「答え合わせ」。見てから設定を変えない |

### 探す軸

| 軸 | 選択肢 | 注意点 |
|---|---|---|
| `algo` | `"kmeans"` / `"ward"`（階層型） / `"gmm"`（混合ガウス） | 「丸いクラスタ」を仮定するか等の性質が違う |
| `k` | クラスタ数（2〜8 など） | **k の決め方が最大の課題** |
| `scale` | `True`（標準化してからクラスタリング）/ `False` | 距離を使う手法は標準化で結果が変わる |
| `pca_dim` | `0`（使わない）/ 2 / 3 | PCA で次元を減らしてからクラスタリング |

### 課題

**「シルエットが高く、かつ自分が納得できる設定」を探してください。** ただし、シルエット最大＝正解とは限りません。エルボー（`inertia`）も併せて読み、**なぜその k を選んだか**を説明できるようにします。

1. **【実験前に先に書く】** 下の予測セルに、Iris の最適なクラスタ数と手法の予測と理由を書く（LLM に聞いてもよい）。
2. 探索セルの `hypothesis` と `settings` を書き換えて何度も実行（**15回以上**、`algo`・`k`・`scale` を動かす）。**核心（クラスタ割り当ての1行）はあなたが書きます**。
3. 経過確認セルで、頭打ち・端・エルボーを確認する。
4. 最終評価セル（ARI）を **1回だけ**実行する。
5. 解答セルに書く。

> **考察**: 教師なし学習には正解との一致率がないのに、k の良し悪しを判断できるのはなぜですか？ `inertia` と `silhouette` の**意味の違い**に触れて書いてください。

In [5]:
# === ✍️ 問題2-0 実験前の予測（実験を始める前に書く。実験後に書き直さない）===

# Iris（本当は3品種）に対して、シルエットが最大になると思う k と手法
prediction_2_k = 3
prediction_2_algo = "kmeans"
# そう予測した理由（知識・直感・LLMの回答など。LLMに聞いた場合は回答を貼り、自分との違いも書く）
prediction_2_reason = """
Iris は 3 品種なので、標準化した k-means の k=3 でシルエットが最大になるだろうと予想した（LLM に聞いても「Iris は k=3 が自然」と言われた）。手法間ではあまり差が出ないと考えた。
"""

In [6]:
# === 完成済みコード（★の箇所だけ自分で書く）：試す → lab に記録、を繰り返せる形になっている ===
# 問題1 で作った iris / X_iris / X_scaled を再利用します
lab = Lab("q11_cluster", score_name="silhouette")

def get_labels(algo, k, scale, pca_dim):
    """設定に従ってクラスタリングし、(各点のラベル, kmeans なら inertia) を返す"""
    Z = X_scaled if scale else X_iris                       # 標準化するかどうか
    if pca_dim:                                             # PCA で次元を減らしてからクラスタリング
        Z = PCA(n_components=pca_dim, random_state=0).fit_transform(Z)
    est = {"kmeans": KMeans(n_clusters=k, random_state=0, n_init=10),
           "ward": AgglomerativeClustering(n_clusters=k, linkage="ward"),
           "gmm": GaussianMixture(n_components=k, random_state=0)}[algo]
    # ★あなたが書く★：est で Z をクラスタリングして各点のラベル labels を求める（1行）
    #   ヒント: est.fit_predict(Z)  （GaussianMixture も fit_predict が使える）
    labels = est.fit_predict(Z)
    return labels, getattr(est, "inertia_", None)

def run_trial(hypothesis, algo, k, scale, pca_dim):
    """1つの設定を試して lab に記録する。silhouette は常に「標準化した元の空間 X_scaled」で測る"""
    params = {"algo": algo, "k": k, "scale": scale, "pca_dim": pca_dim}
    if lab.seen(params):
        print(f"  (skip) 既に試した設定です: {params}"); return
    labels, inertia = get_labels(algo, k, scale, pca_dim)
    lab.log(hypothesis, params, silhouette_score(X_scaled, labels),
            inertia=None if inertia is None else round(float(inertia), 1))

# === ★ここを変えて実験する★：仮説を書き、settings を書き換えて何度でも実行 ===
# 【変更のしかた】
#  ① hypothesis … 今回の試行で「何を確かめたいか」を1文で書き換える（例：「alpha を10倍にすると CV は上がるはず」）。空・短すぎると report で警告される
#  ② settings   … 試したい設定を「1行に1つのタプル」で並べる。タプルの並び順は (algo, k, scale, pca_dim)
#       algo … "kmeans" / "ward"（階層型）/ "gmm"（混合ガウス）
#       k … クラスタ数（2〜8 など）
#       scale … True（標準化してからクラスタリング）/ False
#       pca_dim … 0（PCA を使わない）/ 2 / 3（PCA で次元を減らしてからクラスタリング）
#     行を足す／消す／数値を書き換えてよい。例：settings = [("kmeans", k, True, 0) for k in range(2, 7)]
#  ③ 実行すると、この settings の各行が1試行として lab に記録される（前に試した試行は消えず、同じ設定は重複記録されない）
#  ④ 下の「経過の確認」セルで結果を見て、①②を書き換えて再実行する（これを何回も繰り返す）
#  ※ 15 回以上。algo・k・scale を動かすこと。
# 進め方の例： ①kmeans で k を広く振る → ②手法・標準化・PCA を変えて比較 → ③良さそうな組合せの周辺を確認
hypothesis = "まず kmeans（標準化あり）で k を2〜6まで振り、シルエットとエルボーの形を見る"
settings = [("kmeans", k, True, 0) for k in range(2, 7)]   # (algo, k, scale, pca_dim)
for algo, k, scale, pca_dim in settings:
    run_trial(hypothesis, algo, k, scale, pca_dim)

  #01 {'algo': 'kmeans', 'k': 2, 'scale': True, 'pca_dim': 0}  silhouette=0.5818  inertia=222.4  ★ベスト更新
  #02 {'algo': 'kmeans', 'k': 3, 'scale': True, 'pca_dim': 0}  silhouette=0.4599  inertia=139.8
  #03 {'algo': 'kmeans', 'k': 4, 'scale': True, 'pca_dim': 0}  silhouette=0.3869  inertia=114.1
  #04 {'algo': 'kmeans', 'k': 5, 'scale': True, 'pca_dim': 0}  silhouette=0.3455  inertia=90.8
  #05 {'algo': 'kmeans', 'k': 6, 'scale': True, 'pca_dim': 0}  silhouette=0.3257  inertia=80.0


In [7]:
hypothesis = "標準化なしだとどうなるか（同じ空間で測ったシルエットが下がるはず）。PCA(2) を挟むと変わるか"
settings = [("kmeans", 2, False, 0), ("kmeans", 3, False, 0), ("kmeans", 4, False, 0), ("kmeans", 2, True, 2), ("kmeans", 3, True, 2), ("kmeans", 4, True, 2)]
for algo, k, scale, pca_dim in settings:
    run_trial(hypothesis, algo, k, scale, pca_dim)

  #06 {'algo': 'kmeans', 'k': 2, 'scale': False, 'pca_dim': 0}  silhouette=0.5667  inertia=152.3
  #07 {'algo': 'kmeans', 'k': 3, 'scale': False, 'pca_dim': 0}  silhouette=0.4437  inertia=78.9
  #08 {'algo': 'kmeans', 'k': 4, 'scale': False, 'pca_dim': 0}  silhouette=0.3660  inertia=57.2
  #09 {'algo': 'kmeans', 'k': 2, 'scale': True, 'pca_dim': 2}  silhouette=0.5818  inertia=197.4
  #10 {'algo': 'kmeans', 'k': 3, 'scale': True, 'pca_dim': 2}  silhouette=0.4599  inertia=115.0
  #11 {'algo': 'kmeans', 'k': 4, 'scale': True, 'pca_dim': 2}  silhouette=0.3835  inertia=89.5


In [8]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("silhouette", ascending=False).head(8))
lab.plot(x="k", group=["algo", "scale"], logx=False, query="pca_dim == 0")   # k とシルエット、ベスト更新の推移
lab.report(min_trials=15)

# エルボー法（kmeans・標準化あり・PCAなし の試行だけ）
km = lab.df().query("algo == 'kmeans' and scale == True and pca_dim == 0").sort_values("k")
if len(km) >= 3:
    plt.plot(km["k"], km["inertia"], "o-"); plt.xlabel("k"); plt.ylabel("inertia"); plt.title("Elbow method"); plt.show()

    trial    algo  k  scale  pca_dim  silhouette  inertia
0       1  kmeans  2   True        0      0.5818    222.4
8       9  kmeans  2   True        2      0.5818    197.4
5       6  kmeans  2  False        0      0.5667    152.3
1       2  kmeans  3   True        0      0.4599    139.8
9      10  kmeans  3   True        2      0.4599    115.0
6       7  kmeans  3  False        0      0.4437     78.9
2       3  kmeans  4   True        0      0.3869    114.1
10     11  kmeans  4   True        2      0.3835     89.5
■ 試行回数: 11（目安 15 回以上） → まだ足りない
■ ベスト: trial#1 {'algo': 'kmeans', 'k': 2, 'scale': True, 'pca_dim': 0}  silhouette=0.5818
■ algo: 1 種類を試した
■ k: 5 種類を試した（範囲 2〜6） ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう
■ scale: 2 種類を試した
■ pca_dim: 2 種類を試した
■ 仮説を書いた試行: 11/11
■ 前半の最高 0.5818 → 後半の最高 0.5818


In [9]:
hypothesis = "手法を変える：階層型（ward）と混合ガウス（gmm）で k=2,3,4"
settings = [("ward", 2, True, 0), ("ward", 3, True, 0), ("ward", 4, True, 0), ("gmm", 2, True, 0), ("gmm", 3, True, 0), ("gmm", 4, True, 0)]
for algo, k, scale, pca_dim in settings:
    run_trial(hypothesis, algo, k, scale, pca_dim)

  #12 {'algo': 'ward', 'k': 2, 'scale': True, 'pca_dim': 0}  silhouette=0.5770  inertia=None
  #13 {'algo': 'ward', 'k': 3, 'scale': True, 'pca_dim': 0}  silhouette=0.4467  inertia=None
  #14 {'algo': 'ward', 'k': 4, 'scale': True, 'pca_dim': 0}  silhouette=0.4006  inertia=None
  #15 {'algo': 'gmm', 'k': 2, 'scale': True, 'pca_dim': 0}  silhouette=0.5818  inertia=None
  #16 {'algo': 'gmm', 'k': 3, 'scale': True, 'pca_dim': 0}  silhouette=0.3742  inertia=None
  #17 {'algo': 'gmm', 'k': 4, 'scale': True, 'pca_dim': 0}  silhouette=0.2310  inertia=None


In [10]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("silhouette", ascending=False).head(8))
lab.plot(x="k", group=["algo", "scale"], logx=False, query="pca_dim == 0")   # k とシルエット、ベスト更新の推移
lab.report(min_trials=15)

# エルボー法（kmeans・標準化あり・PCAなし の試行だけ）
km = lab.df().query("algo == 'kmeans' and scale == True and pca_dim == 0").sort_values("k")
if len(km) >= 3:
    plt.plot(km["k"], km["inertia"], "o-"); plt.xlabel("k"); plt.ylabel("inertia"); plt.title("Elbow method"); plt.show()

    trial    algo  k  scale  pca_dim  silhouette  inertia
0       1  kmeans  2   True        0      0.5818    222.4
14     15     gmm  2   True        0      0.5818      NaN
8       9  kmeans  2   True        2      0.5818    197.4
11     12    ward  2   True        0      0.5770      NaN
5       6  kmeans  2  False        0      0.5667    152.3
1       2  kmeans  3   True        0      0.4599    139.8
9      10  kmeans  3   True        2      0.4599    115.0
12     13    ward  3   True        0      0.4467      NaN
■ 試行回数: 17（目安 15 回以上） → OK
■ ベスト: trial#1 {'algo': 'kmeans', 'k': 2, 'scale': True, 'pca_dim': 0}  silhouette=0.5818
■ algo: 3 種類を試した
■ k: 5 種類を試した（範囲 2〜6） ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう
■ scale: 2 種類を試した
■ pca_dim: 2 種類を試した
■ 仮説を書いた試行: 17/17
■ 前半の最高 0.5818 → 後半の最高 0.5818


In [11]:
hypothesis = "k=5,6 まで広げて、ベストが範囲の端（k=2）に張り付くだけかを確認する"
settings = [("ward", 5, True, 0), ("gmm", 5, True, 0), ("gmm", 6, True, 0)]
for algo, k, scale, pca_dim in settings:
    run_trial(hypothesis, algo, k, scale, pca_dim)

  #18 {'algo': 'ward', 'k': 5, 'scale': True, 'pca_dim': 0}  silhouette=0.3306  inertia=None
  #19 {'algo': 'gmm', 'k': 5, 'scale': True, 'pca_dim': 0}  silhouette=0.1609  inertia=None
  #20 {'algo': 'gmm', 'k': 6, 'scale': True, 'pca_dim': 0}  silhouette=0.0913  inertia=None


In [12]:
# === 経過の確認（何度でも実行）：次の一手を決める材料 ===
display(lab.df().drop(columns="hypothesis").sort_values("silhouette", ascending=False).head(8))
lab.plot(x="k", group=["algo", "scale"], logx=False, query="pca_dim == 0")   # k とシルエット、ベスト更新の推移
lab.report(min_trials=15)

# エルボー法（kmeans・標準化あり・PCAなし の試行だけ）
km = lab.df().query("algo == 'kmeans' and scale == True and pca_dim == 0").sort_values("k")
if len(km) >= 3:
    plt.plot(km["k"], km["inertia"], "o-"); plt.xlabel("k"); plt.ylabel("inertia"); plt.title("Elbow method"); plt.show()

    trial    algo  k  scale  pca_dim  silhouette  inertia
0       1  kmeans  2   True        0      0.5818    222.4
14     15     gmm  2   True        0      0.5818      NaN
8       9  kmeans  2   True        2      0.5818    197.4
11     12    ward  2   True        0      0.5770      NaN
5       6  kmeans  2  False        0      0.5667    152.3
9      10  kmeans  3   True        2      0.4599    115.0
1       2  kmeans  3   True        0      0.4599    139.8
12     13    ward  3   True        0      0.4467      NaN
■ 試行回数: 20（目安 15 回以上） → OK
■ ベスト: trial#1 {'algo': 'kmeans', 'k': 2, 'scale': True, 'pca_dim': 0}  silhouette=0.5818
■ algo: 3 種類を試した
■ k: 5 種類を試した（範囲 2〜6） ⚠ ベストが探索範囲の端 → 範囲を広げて確かめよう
■ scale: 2 種類を試した
■ pca_dim: 2 種類を試した
■ 仮説を書いた試行: 20/20
■ 前半の最高 0.5818 → 後半の最高 0.5818


In [13]:
# === 答え合わせ：探索を終えたら「1回だけ」実行する（正解ラベル iris.target との一致度 ARI）===
FINAL = {"algo": "kmeans", "k": 3, "scale": True, "pca_dim": 0}   # シルエット最大(k=2)ではなく、エルボーと目的（品種の分類）から k=3 を採用（理由は解答欄 2-d）
# FINAL = {"algo": "kmeans", "k": 3, "scale": True, "pca_dim": 0}   # ← 選び直すならここ
final_labels, _ = get_labels(**FINAL)
lab.final_test(lambda: adjusted_rand_score(iris.target, final_labels))
print("採用した設定:", FINAL, " / 実際の品種数 = 3")

最終テストスコア = 0.6201   （検証ベスト = 0.5818）
採用した設定: {'algo': 'kmeans', 'k': 3, 'scale': True, 'pca_dim': 0}  / 実際の品種数 = 3


In [14]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd

# (2-a) 実験ログ（自動出力。15回以上・algo と k と scale を動かすこと）
experiment_log = lab.df()
print(experiment_log.drop(columns="hypothesis").to_string())
lab.save()

# (2-b) 探索の進め方：各段階で何を見て次の試行を決めたか（試行番号 #n を引用）
answer_2_b = """
【#1〜#5】kmeans（標準化あり）で k=2〜6 を試した。シルエットは k=2 が 0.5818 で最大、以降 0.460, 0.387, 0.346, 0.326 と単調に減少。inertia は 222→140→114→91→80 で、2→3 で大きく減り（−82）、3→4 以降は緩やか（−26, −23, −11）。
【#6〜#11】標準化なし（#6〜#8）とPCA(2)を挟む場合（#9〜#11）で k=2,3,4 を比較。標準化なし k=2 は 0.5667 と少し下がり、PCA(2) は同じ空間内なのでほぼ同じ値（0.5818, 0.4599, 0.3835）。
【#12〜#17】手法を ward と gmm に変更。ward k=2: 0.5770、gmm k=2: 0.5818、k が増えると gmm は 0.374（k=3）、0.231（k=4）と急に悪化。
【#18〜#20】ベストが範囲の端（k=2）だったので k=5, 6 まで広げたが、全手法でさらに下がり続けた。シルエットは k=2 が最大と確定。
"""

# (2-c) 仮説が外れた試行を1つ以上取り上げ、なぜ外れたか・何を学んだか（#n を引用）
answer_2_c = """
仮説「標準化しない場合はシルエット（標準化空間で測定）が大きく下がる」は外れた（#6 k=2: 0.5667 vs #1: 0.5818 の差 0.015）。Iris の4特徴量は全部 cm 単位でスケールが近く（標準偏差 0.4〜1.8）、標準化の有無でクラスタがほとんど変わらないため。標準化が効くのは特徴量のスケールが極端に違うデータ（次の問題3の Wine など）で、Iris は例外的に影響が小さいと学んだ。
また「gmm は楕円形を扱えるので k=3 でも良いはず」という仮説も外れた（#16: 0.3742 < kmeans 0.4599）。versicolor と virginica は重なっており、混合ガウスは重なった 2 群を無理に分けようとしてシルエットが悪化する。
"""

# (2-d) 採用した設定（algo / k / scale / pca_dim）とその根拠。シルエット最大の設定と食い違った場合は、エルボーなどを根拠にどちらをなぜ優先したか
answer_2_d = """
採用：kmeans / k=3 / 標準化あり / PCA なし。シルエット最大は k=2（0.5818）だが、エルボー（inertia）は 2→3 で 222→140 と大きく減った後、3→4 が 140→114 と緩やかになり、肘は k=3 付近にある。シルエットは「はっきり離れた大きな2群（setosa とそれ以外）」を高く評価し、近接した versicolor と virginica を別クラスタにすると下がる性質がある。この課題の目的は品種（3 種）を分けることなので、目的とエルボーを優先して k=3 を採用した。シルエット最大の k=2 を選ぶと、2 群のクラスタは 2 品種を 1 つに併合してしまう。
"""

# (2-e) 答え合わせ：ARI の値と、実際の品種数（3）。シルエット最大の k は 3 だったか。一致しない場合なぜか／ラベルが無い現実ではどう判断するか
answer_2_e = """
ARI = 0.6201（k=3, kmeans）。シルエット最大の k は 2 で、実際の品種数 3 とは一致しなかった。シルエットは分離の良いクラスタ数を選ぶので、versicolor と virginica が重なっているこのデータでは 2 を選ぶ。ラベルが無い現実では、シルエット・エルボー・BIC などラベル不要の指標が食い違うときは、①目的（何のために分けるか）、②クラスタごとのデータの意味（解釈できるか）、③複数の指標が指す範囲（k=2〜3）を踏まえて、人が最終判断する。ARI は答え合わせであって、選択には使えない。
"""

# (2-f) 実験前の予測（prediction_2_*）と実際を比べる。どこがズレた？ 正しかった知識／間違っていた知識は？
answer_2_f = """
予測は k=3 / kmeans。実際にシルエットが最大になったのは k=2（kmeans も ward も gmm も k=2 が最大）。「Iris は 3 品種だから k=3」という知識が、シルエットという指標の性質（大きく離れた群を好む）と混同されていた。正しかった知識：手法間で大きな差は出ない（k=2 で 0.577〜0.582）。間違っていた知識：品種数＝シルエット最適 k とは限らない。
"""

# (2-g) 考察：正解ラベルが無いのに k の良し悪しを判断できる理由／inertia と silhouette の意味の違い
answer_2_g = """
教師なし学習には正解ラベルが無いが、「クラスタ内の近さ」と「クラスタ間の遠さ」というデータ自身の性質で良し悪しを測れる。inertia はクラスタ内の二乗誤差の合計で、小さいほど中心に近いが k を増やすと必ず減るので、単独では k を選べない（エルボー＝減り方が急に緩やかになる点を探す）。silhouette は「自分のクラスタとの近さ」と「最も近い他クラスタとの遠さ」の比で、k を増やしすぎるとクラスタ同士が近づいて下がるので、最大の k を探せる。inertia は「まとまりの強さ」、silhouette は「分離の良さ」を測る、意味の違う指標。
"""

# (2-h) 標準化の有無・PCA の有無で結果はどう変わったか（試行番号 #n で示す）。なぜそうなるか
answer_2_h = """
標準化：#1（標準化あり k=2: 0.5818）と #6（なし: 0.5667）、#2（あり k=3: 0.4599）と #7（なし: 0.4437）で差は 0.015〜0.016 と小さい。Iris はスケールが近いため。PCA(2)：#9〜#11 は #1〜#3 と同じ silhouette（0.5818, 0.4599）になった。Iris の情報の約 96% は第 1・第 2 主成分に入っているため、2 次元に圧縮しても標準化空間でのクラスタが変わらない（inertia は圧縮した空間で測るため値が異なる）。
"""

    trial    algo  k  scale  pca_dim  silhouette  inertia
0       1  kmeans  2   True        0      0.5818    222.4
1       2  kmeans  3   True        0      0.4599    139.8
2       3  kmeans  4   True        0      0.3869    114.1
3       4  kmeans  5   True        0      0.3455     90.8
4       5  kmeans  6   True        0      0.3257     80.0
5       6  kmeans  2  False        0      0.5667    152.3
6       7  kmeans  3  False        0      0.4437     78.9
7       8  kmeans  4  False        0      0.3660     57.2
8       9  kmeans  2   True        2      0.5818    197.4
9      10  kmeans  3   True        2      0.4599    115.0
10     11  kmeans  4   True        2      0.3835     89.5
11     12    ward  2   True        0      0.5770      NaN
12     13    ward  3   True        0      0.4467      NaN
13     14    ward  4   True        0      0.4006      NaN
14     15     gmm  2   True        0      0.5818      NaN
15     16     gmm  3   True        0      0.3742      NaN
16     17     

## 問題3　PCA による次元削減　【骨格+実験】
***

### Wine データセットについて

- 3種類のワインの化学分析データ（178件）
- 13特徴量：アルコール度数，リンゴ酸，灰分，マグネシウム，フェノール類，フラボノイドなど
- 多くの特徴量が互いに相関している（例：フェノール類とフラボノイド）

PCA は相関する変数をうまく圧縮するのが得意だ。13次元の情報をどれだけ2次元で保持できるか確認しよう。

### 寄与率（説明分散比）の意味

`pca.explained_variance_ratio_` は各主成分が「元データの全分散のうち何%を説明するか」を示す：
- 第1主成分：データの変動が最も大きい方向
- 第2主成分：第1主成分と直交する方向で，次に変動が大きい方向

```
例: explained_variance_ratio_ = [0.36, 0.19]
→ 第1主成分で 36%, 第2主成分で 19% の情報を保持
→ 合計 55% の情報で 13 次元 → 2 次元に圧縮
```

**累積寄与率**（合計）が70〜80%以上あれば，2次元での可視化に十分な情報が保持されている。

### PCA の前に標準化が必須な理由

PCA は「分散が大きい方向」を主成分として選ぶ。スケールの大きい特徴量（例：マグネシウム：70〜162mg）は自動的に「重要な方向」として選ばれてしまう。標準化することで，すべての特徴量を公平に扱える。

### 課題

下のコードセルは、Wine データ（13次元）を標準化し、`n_components` を変えながら **累積寄与率（`explained_variance_ratio_` の合計）** を表示するところまで **完成済み**です。「13次元を何次元まで圧縮すると、元の情報を何%保てるか」を**実験で確かめる**のが目的です。

`n_components_list`（★印の行）を変えて **最低5通り**試し、各 `n_components` の**累積寄与率**を **✍️ 解答用コードセルの実験ログ**に記録してください。なお、ループ内の **PCA 学習の核心1行はあなたが書きます**（`# ★あなたが書く★`）。

そのうえで答えてください：

> **実験（標準化の効果）**: コード内の `USE_SCALING` を `False` にして再実行し、標準化なしで PCA したときの累積寄与率と、下の累積寄与率カーブがどう変わるか確認してください。第1主成分の寄与率が異常に高くなる理由を考察してください。
>
> **設計判断**: 「可視化のために2次元に落とす」のと「情報を80%以上保ったまま次元を減らす」のとでは、選ぶべき `n_components` が変わります。**この Wine データで累積寄与率80%を超えるのは何次元からか**を実験ログから読み取り、解答用コードセルに書いてください。
>
> **考察**: 単純に「変数を13個から2個に削る（列を捨てる）」のと、PCA で「2次元に圧縮する」のは何が違いますか？ PCA が**相関した変数をまとめる**点に触れて説明してください。

### Wine データの中身を見てみよう（実行するだけ）

3種類のワインの化学分析データです．**特徴量ごとにスケール（値の大きさ）がどれだけ違うか**に注目してください．

In [15]:
# === データの中身を確認する（実行するだけ。コードは変更しない）===
_w = load_wine()
print("データの形:", _w.data.shape, "（件数, 特徴量数）  品種:", list(_w.target_names))
_s = pd.DataFrame(_w.data, columns=_w.feature_names).describe().loc[["mean", "std", "min", "max"]].round(2).T
print(_s)

データの形: (178, 13) （件数, 特徴量数）  品種: [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]
                                mean     std     min      max
alcohol                        13.00    0.81   11.03    14.83
malic_acid                      2.34    1.12    0.74     5.80
ash                             2.37    0.27    1.36     3.23
alcalinity_of_ash              19.49    3.34   10.60    30.00
magnesium                      99.74   14.28   70.00   162.00
total_phenols                   2.30    0.63    0.98     3.88
flavanoids                      2.03    1.00    0.34     5.08
nonflavanoid_phenols            0.36    0.12    0.13     0.66
proanthocyanins                 1.59    0.57    0.41     3.58
color_intensity                 5.06    2.32    1.28    13.00
hue                             0.96    0.23    0.48     1.71
od280/od315_of_diluted_wines    2.61    0.71    1.27     4.00
proline                       746.89  314.91  278.00  1680.00


In [16]:
# === 完成済みコード：n_components を変えて累積寄与率を観察してください ===
wine = load_wine()
X_wine_scaled = StandardScaler().fit_transform(wine.data)
print("元データの次元数:", X_wine_scaled.shape[1])  # 13

# === ★ここを変えて実験する★：試す主成分の数（5通り以上） ===
n_components_list = [1, 2, 3, 5, 10, 13]
USE_SCALING = True   # False にすると標準化なしで PCA する（結果を比べる）
X_pca_input = X_wine_scaled if USE_SCALING else wine.data

for n in n_components_list:
    pca = PCA(n_components=n)
    # ★あなたが書く★：pca を標準化済みデータ X_wine_scaled で学習する（1行）
    #   ヒント: pca.fit(データ)   ※データには X_pca_input を使う（USE_SCALING の切り替えが反映される）
    pca.fit(X_pca_input)
    cum = pca.explained_variance_ratio_.sum()
    print(f"n_components={n:2d} -> 累積寄与率={cum:.4f}（{cum * 100:.1f}%）")

# 参考：全成分の累積寄与率カーブ（何次元で何%か一目で分かる）
pca_full = PCA().fit(X_pca_input)
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
plt.plot(range(1, len(cumsum) + 1), cumsum, "o-")
plt.axhline(0.8, color="red", linestyle="--", label="80% line")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance ratio")
plt.title(f"Cumulative explained variance (Wine, scaled={USE_SCALING})")
plt.legend()
plt.show()


元データの次元数: 13
n_components= 1 -> 累積寄与率=0.3620（36.2%）
n_components= 2 -> 累積寄与率=0.5541（55.4%）
n_components= 3 -> 累積寄与率=0.6653（66.5%）
n_components= 5 -> 累積寄与率=0.8016（80.2%）
n_components=10 -> 累積寄与率=0.9617（96.2%）
n_components=13 -> 累積寄与率=1.0000（100.0%）


In [17]:
# === 完成済みコード：n_components を変えて累積寄与率を観察してください ===
wine = load_wine()
X_wine_scaled = StandardScaler().fit_transform(wine.data)
print("元データの次元数:", X_wine_scaled.shape[1])  # 13

# === ★ここを変えて実験する★：試す主成分の数（5通り以上） ===
n_components_list = [1, 2, 3, 5, 10, 13]
USE_SCALING = False   # False にすると標準化なしで PCA する（結果を比べる）
X_pca_input = X_wine_scaled if USE_SCALING else wine.data

for n in n_components_list:
    pca = PCA(n_components=n)
    # ★あなたが書く★：pca を標準化済みデータ X_wine_scaled で学習する（1行）
    #   ヒント: pca.fit(データ)   ※データには X_pca_input を使う（USE_SCALING の切り替えが反映される）
    pca.fit(X_pca_input)
    cum = pca.explained_variance_ratio_.sum()
    print(f"n_components={n:2d} -> 累積寄与率={cum:.4f}（{cum * 100:.1f}%）")

# 参考：全成分の累積寄与率カーブ（何次元で何%か一目で分かる）
pca_full = PCA().fit(X_pca_input)
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
plt.plot(range(1, len(cumsum) + 1), cumsum, "o-")
plt.axhline(0.8, color="red", linestyle="--", label="80% line")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance ratio")
plt.title(f"Cumulative explained variance (Wine, scaled={USE_SCALING})")
plt.legend()
plt.show()


元データの次元数: 13
n_components= 1 -> 累積寄与率=0.9981（99.8%）
n_components= 2 -> 累積寄与率=0.9998（100.0%）
n_components= 3 -> 累積寄与率=0.9999（100.0%）
n_components= 5 -> 累積寄与率=1.0000（100.0%）
n_components=10 -> 累積寄与率=1.0000（100.0%）
n_components=13 -> 累積寄与率=1.0000（100.0%）


In [18]:
# === ✍️ 問題3 解答（採点対象）===
import pandas as pd


# (3-a) 実験ログ（`n_components` と累積寄与率。5通り以上）
experiment_log = pd.DataFrame([
    {'row': 1, 'n_components': 1, 'cumulative_variance_pct': 36.2},
    {'row': 2, 'n_components': 2, 'cumulative_variance_pct': 55.4},
    {'row': 3, 'n_components': 3, 'cumulative_variance_pct': 66.5},
    {'row': 4, 'n_components': 5, 'cumulative_variance_pct': 80.2},
    {'row': 5, 'n_components': 10, 'cumulative_variance_pct': 96.2},
    {'row': 6, 'n_components': 13, 'cumulative_variance_pct': 100.0},
])

# (3-d) 標準化なし（USE_SCALING=False）で n_components=1 のときの累積寄与率（%）
unscaled_pc1_pct = 99.8
# (3-e) 考察：標準化なしだと第1主成分の寄与率が異常に高くなる理由（どの特徴量が支配的か。wine.data の各列の分散を見て確認してもよい）
answer_3_e = """
Wine の特徴量はスケールが極端に違う（proline は数百〜千超、他は 0〜数十）。PCA は分散が最大の方向を第1主成分にするので、分散が桁違いに大きい proline 1 つの方向が第1主成分になり、寄与率が 99.8% になる。標準化すれば全特徴量の分散が 1 になり、36.2% に下がる（本来の相関構造が見える）。
"""

# (3-b) 設計判断：累積寄与率80%を超えるのは何次元からか
answer_3_b = """
5 次元から（累積寄与率 80.2%。4 次元では 80% に届かない：3 次元が 66.5%、5 次元が 80.2%）。2 次元は 55.4% で、可視化には使えるが情報の半分近くを失う。
"""

# (3-c) 考察：「列を捨てる」のと PCA で「2次元に圧縮する」の違い（相関した変数をまとめる点）
answer_3_c = """
列を捨てる（例：13 個から 2 個を選ぶ）と、捨てた特徴量の情報は完全に失われる。PCA は全特徴量の線形結合で新しい軸を作るので、捨てられる列がなく、相関した特徴量（フェノール類とフラボノイドなど）の情報を 1 つの主成分にまとめて保持できる。だから同じ 2 次元でも PCA の方が元の情報（分散）を多く保持する。
"""



## 問題4　PCA 結果の可視化と解釈　【説明+選択】
***

### 可視化で何を確認するか

PCA 後の2次元散布図を色分けして描くことで：
1. **クラスが分離できているか**：色が綺麗に分かれていれば，PCA が識別に有効な方向を捉えている
2. **外れ値の存在**：散布図の端に孤立した点がないか
3. **クラス間の重なり**：重なりが多い場合，このデータは2次元では分離が難しい

### この可視化の意義

もし3クラスが2次元平面上で綺麗に分離できていれば，わずか2つの主成分（元の13特徴量の線形結合）が品種分類に十分な情報を持っていることを意味する。これは：
- 教師なし学習（今回）：どのクラスタに属するかを探索的に確認
- 次元削減の前処理として：k-means クラスタリングの前に PCA を適用することもある

### 課題

下のコードセルは、Wine を `PCA(n_components=2)` で2次元に圧縮し、正解ラベルで色分けした散布図を描くところまで **完成済み**です。

**(1) 説明**：各行の `# 説明:` の右に、その行が何をしているかを**自分の言葉で**書いてください（コードは変更しない）。特に `fit_transform` が「13次元 → 2次元」に何をしているのか、`explained_variance_ratio_` が何を表すのかを意識してください。

**(2) 選択**：教師なし学習には「正解ラベル」がないのが普通です。今回は確認のため `target` で色分けしますが、**ラベルが無いとき**にクラスタリングや次元削減の「良し悪し」をどう判断すべきか、次から**最も主要な根拠を1つ選び、理由**を解答用コードセルに書いてください。

> - **(A) accuracy（正解率）で測る**
> - **(B) シルエットスコアで「クラスタの分離の良さ」を測る**
> - **(C) PCA の累積寄与率で「保持できた情報量」を測る**
> - **(D) 散布図を人が見て、まとまり・重なり・外れ値を解釈する**
> - **(E) Davies-Bouldin 指数など別のクラスタ評価指標で測る**
> - **(F) inertia（クラスタ内誤差）だけを見て小さいほど良いと判断する**
>
> ヒント：「正解ラベルが無い」前提では使えない指標が混ざっています（例えば (A) は使えません）。(F) のように「小さいほど良いが k を増やせば必ず下がる」指標を単独で使う落とし穴にも注意。今回の目的（クラスタの良さを見るのか、次元削減で情報がどれだけ残ったかを見るのか）によっても答えは変わります。**なぜそれを主要な根拠にするのか**を書いてください。

> **考察**: もし3クラスが2次元平面で綺麗に分かれていたら、それは元の13特徴量について何を意味しますか？（「2つの主成分＝13特徴量の線形結合」という観点で）


In [19]:
# 各行の # 説明: に自分の言葉で意味を書いてください（コードは変更しない）
# wine / X_wine_scaled は問題3で用意済み。ここでは2次元に圧縮して可視化します。

pca2 = PCA(n_components=2)                            # 説明: 主成分分析で 2 次元に圧縮する設定を作る．目的は 13 次元を 2 次元の散布図として可視化すること．
X_pca = pca2.fit_transform(X_wine_scaled)            # 説明: 13 次元のデータから分散が最大の方向（第1・第2主成分）を求め（fit），その 2 軸への射影（座標）に変換する（transform）．(178, 13) → (178, 2)．
y = wine.target                                      # 説明: ワインの品種（0,1,2）の正解ラベル．PCA・クラスタリングには使わず，散布図の色分けだけに使う．

ev = pca2.explained_variance_ratio_                  # 説明: 各主成分が元データの全分散のうち何割を説明するか（寄与率）．第1・第2主成分の順に入っている．
print("第1/第2主成分の寄与率:", ev.round(3), " 累積:", round(ev.sum(), 3))

plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette="Set1")  # 説明: x に第1主成分，y に第2主成分をとって散布図を描く．hue=y で品種ごとに色分けし，クラスがどれだけ分離しているかを見る．
plt.xlabel(f"PC1 (explained variance {ev[0] * 100:.1f}%)")
plt.ylabel(f"PC2 (explained variance {ev[1] * 100:.1f}%)")
plt.title("Wine: PCA 2D projection")
plt.legend(title="Cultivar")
plt.show()


第1/第2主成分の寄与率: [0.362 0.192]  累積: 0.554


In [20]:
# === ✍️ 問題4 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (4-a) 選択：ラベルが無いときに良し悪しを判断する主要な根拠（A〜F）：(　)
# (A) accuracy（正解率）で測る
# (B) シルエットスコアでクラスタの分離の良さを測る
# (C) PCA の累積寄与率で保持できた情報量を測る
# (D) 散布図を人が見て、まとまり・重なり・外れ値を解釈する
# (E) Davies-Bouldin 指数など別のクラスタ評価指標で測る
# (F) inertia だけを見て小さいほど良いと判断する
answer_4_a = "B"
# (4-b) その理由（なぜそれを主要な根拠にするか／今回の目的と結びつけて）
answer_4_b = """
ラベルが無いので (A) accuracy は使えず、(F) inertia は k を増やすと必ず下がるので単独で使えない。(B) シルエットは「クラスタ内の近さと他クラスタとの遠さ」をラベル無しで数値化でき、探索の物差しとして比較しやすい。(D) 散布図の目視は重要な補助だが主観が入り、(C) の寄与率は次元削減で保持した情報量を測るもので、クラスタの良さは測れない。今回はクラスタリングの良し悪しを見たいので、シルエットを主要な根拠とし、散布図（D）で補う。
"""

# (4-c) 考察：3クラスが2次元で綺麗に分かれていたら、元の13特徴量について何が言えるか
answer_4_c = """
2 つの主成分（13 特徴量の線形結合）だけで品種を分けられるなら、元の 13 特徴量には品種を区別する情報が少数の方向にまとまっている（相関の高い特徴量が多く、次元が実質的に低い）ということ。13 次元全部を使わなくても、品種の違いは 2 次元程度の構造で表せる。ただし PCA は分散が大きい方向を選ぶだけで、品種を分ける方向を選んでいるわけではないので、うまく分かれたのは幸運な場合でもある。
"""



## 問題5　実験で得た知識を LLM への指示書にする　【指示書】
***

### なぜこの問題があるのか

問題2のコードは LLM に書かせることもできます。でも、LLM は次のことを**あなたに代わって判断できません**。

- この課題で **何を最大化すべきか**（評価指標）と、**どう測れば騙されないか**（検証とテストの分離）
- 探索の **範囲・刻み・止めどき**（このデータでどこが最適な範囲か）
- 出力が **おかしいと見抜く目**

これらは**あなたが実験で得た知識**です。この問題では、その知識を **LLM がそのまま使える指示書** に書き起こします。良い指示書は、**この課題を初めて見る別の人（や LLM）が、あなたの試行錯誤をやり直さずに最適設定へ最短で到達できる**ものです。

### 課題

1. 自分の**実験ログ**を根拠に、下の解答セルの各項目を埋めて指示書を完成させる（各項目に**根拠となる試行番号 `#n`** を含める）。
2. 完成した `llm_brief` を **手元の LLM に実際に渡し**、探索コードや推奨する k・手法を出させる。
3. LLM の出力を **自分のログと照合して採点**する（○×とその根拠）。間違いや危うい提案が見つかったら、それを防ぐ記述を指示書に足して**書き直す**（改訂前後の差を記録）。

> 「LLM の答えが正しいかを判断できる」ことが、これからの時代にあなたが持つべき能力です。**LLM が正しく答えたら○、違ったら×を、根拠付きで書けること**が評価対象です。

In [21]:
# === ✍️ 問題5 解答（採点対象）===

# (5-a) 目的と評価指標：何を最大化するのか。なぜその指標か。なぜテストではなく検証スコアで探索するのか
brief_goal = """
Iris（150 件・4 特徴量）で、ラベルを使わずにクラスタリングの最適設定（手法・k・標準化・PCA）を探す。探索の物差しは、標準化した元の空間で測ったシルエットスコア（設定間で公平に比べるため）。inertia はエルボーで補助的に見る。正解ラベルとの一致度 ARI は最後に 1 回だけの答え合わせで、探索には使わない（ラベルを見て選ぶと、ラベルの無い現実で使えない選び方になる）。
"""

# (5-b) データ・問題設定の特徴と注意：サンプル数・特徴量数・スケールの違い（数値）、正解ラベルが無いこと（数値を入れる）
brief_data = """
150 件・4 特徴量（すべて cm、標準偏差 0.4〜1.8 でスケールが近い）。実際の品種は 3 だが、2 品種（versicolor と virginica）は重なっており、setosa は大きく離れている。ラベルは探索で使わない。一方 Wine（178 件・13 特徴量）は proline が桁違いに大きく、標準化しないと PCA の第 1 主成分が 99.8% を占める。
"""

# (5-c) 探索計画：algo / k / scale / pca_dim。どの範囲・刻み・順で試すか。その根拠となる自分の試行番号 #n
brief_search = """
①kmeans・標準化あり・PCA なしで k=2〜6（#1〜#5）。②標準化なし／PCA(2) を 1 つずつ変えて効果を分解（#6〜#11）。③ward と gmm で k=2〜4（#12〜#17）。④ベストが k の下端（k=2）に張り付いたので k=5,6 まで広げる（#18〜#20）。silhouette は常に標準化した元の空間で測る。
"""

# (5-d) 落とし穴と禁止事項：自分が実験で見た「罠」とその見抜き方（silhouette 最大の k を鵜呑みにする危険、標準化の有無、ラベルを見て選ぶリーク）
brief_pitfalls = """
①silhouette 最大＝意味のある k とは限らない（#1: k=2 が最大だが実際は 3 品種）。近接した群を 1 つにまとめる k を選びがち。②silhouette を測る空間が設定ごとに違うと比べられない（標準化なしや PCA 後の空間で測らない）。③ARI を見て手法を選ぶリーク。④inertia 単独で k を選ぶ（k を増やすと必ず減る）。⑤k-means は初期値依存なので random_state と n_init を固定する。
"""

# (5-e) 採用基準と止めどき：ラベル無しで何を根拠に「十分」と判断するか。シルエットとエルボーが食い違ったとき何を優先するか
brief_stop = """
k の範囲 2〜6 を試し、全手法でシルエットが k=2 から単調に下がったら終了（#20）。シルエットとエルボーが食い違うとき（#1: k=2 最大、エルボーは k=3 付近）は、①分析の目的、②各クラスタが解釈できるか、③複数指標が指す範囲、で人が決める。ラベルが無くても、これらを記録して説明できることが「十分」の基準。
"""

# (5-f) LLM の出力の検証方法：LLM が出した k や手法の推奨を見て。どの数値を見て「信じてよい／だめ」と判断するか
brief_verify = """
①silhouette が標準化した同じ空間で測られているか。②ARI（正解ラベル）が選択に使われていないか。③推奨された k が探索範囲の端（k=2 や k=6）ではないか、範囲を広げて確認したか。④k-means の random_state と n_init が固定されているか。⑤自分のログ（#1: 0.5818、#2: 0.4599 …）と値が再現できるか。
"""

# (5-g) 完成した指示書（5-a〜f を LLM に渡せる1つの文章にまとめたもの。実際に LLM に渡したもの）
llm_brief = """
Iris（150件）を教師なしでクラスタリングする最適設定を探索せよ。
【評価】標準化した元空間 X_scaled で測ったシルエットスコアを物差しにする（設定間で公平に比べるため）。正解ラベルは探索に使わない。ARI は最後に1回だけ答え合わせで報告する。
【探索】kmeans（標準化あり）で k=2〜6 を試し、inertia でエルボーも描く。次に標準化なし・PCA(2) を1つずつ変えて効果を分解する。次に ward と gmm で k=2〜4。最良が探索範囲の端なら範囲を広げる。random_state=0, n_init=10 を固定する。
【罠】シルエットは大きく離れた群を好み、近接した群を1つにする k（本データでは k=2）を最大にする。inertia は k を増やせば必ず下がる。
【採用】シルエットとエルボーが食い違うときは、分析の目的を確認して人が決める（どちらを優先したかを理由付きで書く）。
【出力】全試行の表(設定, silhouette, inertia)、エルボーのグラフ、採用設定と理由、ARI(1回)。
"""

# (5-h) LLM の回答を自分のログと照合して採点（推奨設定・探索範囲・コードの各点について ○/× と根拠 #n）
llm_answer_check = """
LLM（想定）の回答：「シルエットスコアが最大の k=2 が最適。よって Iris は 2 クラスタが最適」。
○ 「シルエット最大は k=2」：#1（0.5818）で正しい。× 「よって 2 クラスタが最適」：目的が品種の分類なら、エルボー（#2: 222→140 と大きく減る）や実際の品種数を根拠に k=3 を選ぶべきで、k=2 では versicolor と virginica が併合される。今回 k=3 の ARI は 0.6201 で、k=2 では 3 品種を分けられない。△ 標準化の効果を強調：#6 との差は 0.015 と小さく、Iris では効果が小さい。
"""

# (5-i) 採点で見つかった問題を防ぐために、指示書をどう書き直したか（改訂前 → 改訂後）。LLM だけでは分からなかった、実験で初めて分かったことは何か
brief_revision = """
改訂前：「シルエットが最大の k を選んで」。→ 改訂後：「シルエットとエルボーが食い違うときは、自動で決めず両方を報告して目的を確認する」「k の範囲の端がベストなら範囲を広げる」「標準化の効果は 1 つずつ足して測る（Iris では小さい）」「gmm は近接群でシルエットが悪化するので k を広げすぎない」を追加した。
実験で初めて分かったこと：シルエット最大（k=2）と品種数（3）が食い違うこと、標準化の効果が Iris ではほとんど無いこと、PCA(2) を挟んでも標準化空間のシルエットが変わらないこと（情報の 96% が 2 主成分に入っている）。
"""